In [1]:
import sys
from pathlib import Path
import torch
from transformers import (
    AutoModelForSequenceClassification, 
    AutoTokenizer, 
    TrainingArguments,
    DataCollatorWithPadding
)
# Setup Root e Import
ROOT = Path.cwd().resolve().parent
if str(ROOT / "src") not in sys.path:
    sys.path.append(str(ROOT / "src"))

from project_paths import get_paths
from teacher_finetune_headtail import build_teacher_tokenizer
from distillation import (
    TinyBERTPhase1, 
    Phase1DistillationTrainer, 
    patch_model_for_unnormalized_attention
)

# Paths
paths = get_paths(ROOT)
DATA_DIR = paths.data_processed

print(f"Data Source: {DATA_DIR}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

c:\Users\cola0\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Data Source: C:\Users\cola0\Desktop\nlp.project-Colangelo-2526\data\processed
Device: cuda


In [2]:
from datasets import load_dataset


TEACHER_NAME = "bert-base-uncased"
tokenizer = build_teacher_tokenizer(TEACHER_NAME)
collator = DataCollatorWithPadding(tokenizer)


print("Loading from Parquet files...")
train_ds = load_dataset("parquet", data_files=str(DATA_DIR / "train.parquet"))["train"]

print(f"Train size: {len(train_ds)}")

Loading from Parquet files...
Train size: 231423


In [3]:
cols_to_keep = ["input_ids", "attention_mask", "token_type_ids", "labels"]
cols_to_remove = [col for col in train_ds.column_names if col not in cols_to_keep]

train_ds = train_ds.remove_columns(cols_to_remove)


In [4]:
TEACHER_PATH = str(paths.checkpoints / "bert_teacher_finetuned" / "checkpoint-18000") 
STUDENT_NAME = "huawei-noah/TinyBERT_General_4L_312D"

print("Loading Teacher...")
teacher = AutoModelForSequenceClassification.from_pretrained(TEACHER_PATH, attn_implementation="eager")

print("Loading Student Base...")
student_base = AutoModelForSequenceClassification.from_pretrained(STUDENT_NAME, num_labels=2, attn_implementation="eager")

print("Patching models for pre-softmax attention...")
patch_model_for_unnormalized_attention(teacher)
patch_model_for_unnormalized_attention(student_base)

teacher = teacher.to(device)

student_phase1 = TinyBERTPhase1(student_model=student_base).to(device)

Loading Teacher...
Loading Student Base...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at huawei-noah/TinyBERT_General_4L_312D and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Patching models for pre-softmax attention...
Model bert patched: now output_attentions returns pre-softmax logits.
Model bert patched: now output_attentions returns pre-softmax logits.


In [5]:
PHASE1_OUTPUT = paths.checkpoints / "tinybert_phase1"

training_args = TrainingArguments(
    output_dir=str(PHASE1_OUTPUT),
    per_device_train_batch_size=8,      # debug/stabilità
    gradient_accumulation_steps=4, # Alza al massimo che la VRAM ti concede per stabilizzare la MSE
    num_train_epochs=8,             # Jiao et al. suggeriscono molte epoche per questa fase
    learning_rate=5e-5,             # LR costante, NIENTE LLRD qui
    fp16=False,                      # Risparmia VRAM
    #bf16=True,
    logging_steps=500,
    save_strategy="epoch",   
    save_total_limit=2,              # Disabilitiamo i salvataggi automatici per evitare di salvare le matrici di proiezione
    report_to="none",
    logging_nan_inf_filter=False,
    dataloader_num_workers=0,
    warmup_steps=200,
    remove_unused_columns=False     # Cruciale! Altrimenti il Trainer scarta input_ids se non vede "labels" nel forward
)

In [6]:
from transformers import TrainerCallback

class TDLossPrinterCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return
        keys = ["loss", "td_loss_embd", "td_loss_hidn", "td_loss_attn", "td_loss_total_raw", "learning_rate"]
        msg_parts = []
        for k in keys:
            if k == "learning_rate":
                msg_parts.append(f"{k}={logs[k]:.2e}")
            elif isinstance(logs[k], float):
                msg_parts.append(f"{k}={logs[k]:.4f}")
            else:
                msg_parts.append(f"{k}={logs[k]}")
        if msg_parts:
            print(f"[step {state.global_step}] " + " | ".join(msg_parts))

In [ ]:
trainer = Phase1DistillationTrainer(
    teacher_model=teacher,
    model=student_phase1,
    args=training_args,
    train_dataset=train_ds,
    data_collator=collator,
    callbacks=[TDLossPrinterCallback()]
)

print("Starting Intermediate Layer Distillation (Phase 1)...")
trainer.train()

Starting Intermediate Layer Distillation (Phase 1)...


Step,Training Loss
500,21.307100
1000,2.617800
1500,2.372200
2000,2.270900
2500,2.211400
3000,2.173100
3500,2.145200
4000,2.122400
4500,2.105700
5000,2.092700


[step 500] loss=21.3071 | td_loss_embd=0.2209 | td_loss_hidn=0.3212 | td_loss_attn=2.4334 | td_loss_total_raw=2.9755 | learning_rate=4.97e-05
[step 1000] loss=2.6178 | td_loss_embd=0.1513 | td_loss_hidn=0.2564 | td_loss_attn=2.0136 | td_loss_total_raw=2.4213 | learning_rate=4.93e-05
[step 1500] loss=2.3722 | td_loss_embd=0.1347 | td_loss_hidn=0.2332 | td_loss_attn=1.8894 | td_loss_total_raw=2.2572 | learning_rate=4.89e-05
[step 2000] loss=2.2709 | td_loss_embd=0.1322 | td_loss_hidn=0.2170 | td_loss_attn=1.8417 | td_loss_total_raw=2.1909 | learning_rate=4.84e-05
[step 2500] loss=2.2114 | td_loss_embd=0.1316 | td_loss_hidn=0.2206 | td_loss_attn=1.8278 | td_loss_total_raw=2.1801 | learning_rate=4.80e-05
[step 3000] loss=2.1731 | td_loss_embd=0.1239 | td_loss_hidn=0.2156 | td_loss_attn=2.2899 | td_loss_total_raw=2.6294 | learning_rate=4.76e-05
[step 3500] loss=2.1452 | td_loss_embd=0.1282 | td_loss_hidn=0.2059 | td_loss_attn=1.7508 | td_loss_total_raw=2.0849 | learning_rate=4.71e-05
[step 

In [ ]:
final_student_path = PHASE1_OUTPUT / "student_base_final"

print(f"Salvataggio del modello student  in: {final_student_path}")
student_phase1.student.save_pretrained(str(final_student_path))
tokenizer.save_pretrained(str(final_student_path))
print("Salvataggio completato")